In [ ]:
import time
import torch
from utils.dataclass import simulate_for_likelihood
from utils.dataclass import TimeSeriesInferenceDataset

def _sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)

def loglike(self, theta_phys):
    device = self.device

    t0 = time.perf_counter()
    _sync(device)

    # --- sanitize theta ---
    theta_phys = theta_phys.to(device=device, dtype=self.kappa_samples.dtype).reshape(-1)  # see note below
    if theta_phys.numel() != 3:
        raise ValueError("theta_phys must have 3 entries [alpha, mu, d]")

    _sync(device)
    t1 = time.perf_counter()

    # --- build params ---
    params = self._build_params_with_capacity(theta_phys)
    _sync(device)
    t2 = time.perf_counter()

    # --- simulate ---
    ns = simulate_for_likelihood(params, self.times)
    _sync(device)
    t3 = time.perf_counter()

    # --- score ---
    log_probs = self.lpkdil_ns(ns, reduce=True, concat=True)
    _sync(device)
    t4 = time.perf_counter()

    out = torch.sum(log_probs)
    _sync(device)
    t5 = time.perf_counter()

    print(
        f"sanitize {(t1-t0):.3f}s | build {(t2-t1):.3f}s | sim {(t3-t2):.3f}s | "
        f"score {(t4-t3):.3f}s | sum {(t5-t4):.3f}s | total {(t5-t0):.3f}s"
    )

    return out
